### Importing libraries and Loading data

In [ ]:
# Run this BEFORE Cell 1
# Tells us exactly what's wrong!

import sys
print(f"1. Python: {sys.executable}")

try:
    import sentence_transformers
    print(f"2. sentence_transformers installed")
except:
    print(f"2. sentence_transformers MISSING!")

try:
    import faiss
    print(f"3. faiss installed")
except:
    print(f"3. faiss MISSING!")

import os
cf = '../data/processed/chunks.json'
ff = '../vector_store/faiss_index/alzheimer.index'
print(f"5. alzheimer.index: {'✅' if os.path.exists(ff) else '❌'}")

print(f"4. chunks.json: {'✅' if os.path.exists(cf) else '❌'}")

In [ ]:
# Load All Components ────────────
import faiss
import numpy as np
import json
import os
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

# ── Paths ──────────────────────────────────
CHUNKS_FILE = '../data/processed/chunks.json'
FAISS_FILE  = '../vector_store/faiss_index/alzheimer.index'

# ── Load components ────────────────────────
print(" Loading components...")

# Load chunks
with open(CHUNKS_FILE) as f:
    chunks = json.load(f)
print(f" Chunks loaded:    {len(chunks):,}")

# Load FAISS index
index = faiss.read_index(FAISS_FILE)
print(f" FAISS loaded:     {index.ntotal:,} vectors")

# Load model
model = SentenceTransformer(
    'neuml/pubmedbert-base-embeddings'
)
print(f" Model loaded:     PubMedBERT 768-dim")
print(f"\n All components ready!")

In [ ]:
#  Search Function:
def search_papers(query, k=3):
    """
    Search for relevant chunks
    using FAISS semantic search
    """
    # Encode query
    query_vector = model.encode([query])
    query_vector = query_vector.astype('float32')

    # Search FAISS
    distances, indices = index.search(
        query_vector, k=k
    )

    # Get results
    results = []
    for dist, idx in zip(
        distances[0], indices[0]
    ):
        chunk = chunks[idx]
        results.append({
            "title":    chunk['title'],
            "year":     chunk['year'],
            "authors":  chunk['authors'],
            "journal":  chunk['journal'],
            "chunk":    chunk['chunk'],
            "distance": float(dist)
        })

    return results

print("Search function ready!")

### Test Query 1  

In [ ]:
#  Test 1 — Biomarkers ────────────
query = "What blood biomarkers detect Alzheimer's early?"

print(f" Query: {query}")
print(f"{'='*55}")

results = search_papers(query, k=3)

for i, r in enumerate(results):
    print(f"\n Result {i+1}:")
    print(f"   Title:    {r['title'][:60]}...")
    print(f"   Year:     {r['year']}")
    print(f"   Authors:  {r['authors'][:40]}")
    print(f"   Journal:  {r['journal'][:40]}")
    print(f"   Distance: {r['distance']:.2f}")
    print(f"   Preview:  {r['chunk'][:150]}...")

### Test Query 2

In [ ]:
#  Test 2 — Diabetes ──────────────
query = "How does diabetes connect to Alzheimer's disease?"

print(f" Query: {query}")
print(f"{'='*55}")

results = search_papers(query, k=3)

for i, r in enumerate(results):
    print(f"\n Result {i+1}:")
    print(f"   Title:    {r['title'][:60]}...")
    print(f"   Year:     {r['year']}")
    print(f"   Distance: {r['distance']:.2f}")
    print(f"   Preview:  {r['chunk'][:150]}...")

###  Test Query 3

In [ ]:
# Test 3 — Treatment ─────────────
query = "What is Lecanemab and how does it treat Alzheimer's?"

print(f" Query: {query}")
print(f"{'='*55}")

results = search_papers(query, k=3)

for i, r in enumerate(results):
    print(f"\n Result {i+1}:")
    print(f"   Title:    {r['title'][:60]}...")
    print(f"   Year:     {r['year']}")
    print(f"   Distance: {r['distance']:.2f}")
    print(f"   Preview:  {r['chunk'][:150]}...")

### Search Quality Report

In [ ]:
# Search Quality Report ──────────
print(f"\n {'='*55}")
print(f"SEARCH QUALITY REPORT")
print(f"{'='*55}")

test_queries = [
    "blood biomarkers Alzheimer's detection",
    "diabetes insulin resistance brain",
    "Lecanemab treatment amyloid",
    "APOE4 genetic risk dementia",
    "deep learning MRI Alzheimer's"
]

for query in test_queries:
    results = search_papers(query, k=1)
    best    = results[0]
    print(f"\n {query[:45]}...")
    print(f"   Best match: {best['title'][:50]}...")
    print(f"   Year:       {best['year']}")
    print(f"   Distance:   {best['distance']:.2f}")

print(f"\n{'='*55}")
print(f" Search pipeline working!")
print(f"   Index:   {index.ntotal:,} vectors")
print(f"   Model:   PubMedBERT 768-dim")
print(f"   Status:  READY FOR RAG! 🚀")
print(f"{'='*55}")

